# Enhanced Optuna Hyperparameter Search — GraphSAGE 

## Purpose
Takes the four feature subsets from `selected_features.json` (PSO, HHO, MI, ALL) and
runs an Optuna search on GraphSAGE, independently per subset.


Search space unchanged from the prior enhanced search (hidden_dim, dropout, lr,
weight_decay, k_neighbours, n2v_dim, optimiser, scheduler). Node2Vec walk structure
unchanged (WALK_LEN=20, CONTEXT=10, WALKS=10, EPOCHS=50, LR=0.01).


In [5]:
# CELL 1 — INSTALLATION
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args, '-q'], check=False)

pip('torch_geometric', 'numpy>=2')
pip('imbalanced-learn', 'numpy>=2')
pip('optuna')

print("Installation complete. Restart the kernel, then run Cell 2 onward.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:00
Installation complete. Restart the kernel, then run Cell 2 onward.


In [6]:
# CELL 2 — IMPORTS, SEED, CONFIG
import os, gc, json, random, warnings
warnings.filterwarnings('ignore')
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import numpy  as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.nn import Linear

import torch_geometric
from torch_geometric.data  import Data
from torch_geometric.nn    import SAGEConv
from torch_geometric.utils import coalesce
from gensim.models         import Word2Vec

from sklearn.model_selection  import train_test_split, StratifiedKFold
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics          import matthews_corrcoef, roc_auc_score
from imblearn.over_sampling   import SMOTE

print(f"PyTorch           : {torch.__version__}")
print(f"PyTorch Geometric : {torch_geometric.__version__}")
print(f"Optuna            : {optuna.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

GLOBAL_SEED = 42
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(GLOBAL_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device            : {DEVICE}")

# Base configuration
BASE_CONFIG = {
    'DATA_PATH'    : ('/kaggle/input/datasets/monamehrun/'
                      'pcos-cleaned-dataset/pcos_cleaned.csv'),
    'SELECTED_FEATURES_PATH' : '/kaggle/input/datasets/galibbhai/selected-features/selected_features.json',
    'TARGET_COL'   : 'PCOS',
    'OUTPUT_DIR'   : '/kaggle/working/',
    'TEST_SIZE'    : 0.20,          
    'INNER_VAL'    : 0.20,          
    'SEED'         : GLOBAL_SEED,

    'N2V_WALK_LEN' : 20,
    'N2V_CONTEXT'  : 10,
    'N2V_WALKS'    : 10,
    'N2V_EPOCHS'   : 50,
    'N2V_LR'       : 0.01,

    'TRIAL_EPOCHS' : 150,      
    'INNER_FOLDS'  : 5,        
}

TRIALS_PER_SUBSET = {'PSO': 50, 'HHO': 50, 'MI': 30, 'ALL': 30}

print("\nBase configuration loaded.")
print(f"Trial epochs      : {BASE_CONFIG['TRIAL_EPOCHS']} (fixed, no early stopping)")
print(f"Inner CV          : {BASE_CONFIG['INNER_FOLDS']}-fold StratifiedKFold")
print(f"Trials per subset : {TRIALS_PER_SUBSET}")

PyTorch           : 2.10.0+cu128
PyTorch Geometric : 2.8.0
Optuna            : 4.9.0
CUDA available    : True
Device            : cuda

Base configuration loaded.
Trial epochs      : 150 (fixed, no early stopping)
Inner CV          : 5-fold StratifiedKFold
Trials per subset : {'PSO': 50, 'HHO': 50, 'MI': 30, 'ALL': 30}


In [7]:
# CELL 3 — GRAPH, NODE2VEC AND SMOTE UTILITIES

def build_knn_graph(features_scaled: np.ndarray, k: int = 10):
    n   = len(features_scaled)
    sim = cosine_similarity(features_scaled)
    np.fill_diagonal(sim, -2.0)
    src, dst, wts = [], [], []
    for i in range(n):
        top_k = np.argpartition(sim[i], -k)[-k:]
        for j in top_k:
            w = float(max(0.0, sim[i][j]))
            src += [i, j];  dst += [j, i];  wts += [w, w]
    ei = torch.tensor([src, dst], dtype=torch.long)
    ew = torch.tensor(wts,        dtype=torch.float)
    ei, ew = coalesce(ei, ew, num_nodes=n, reduce='max')
    return ei, ew


def add_synthetic_nodes(ei, ew, X_real_sc, X_syn_sc, k=10):
    n_real, n_syn = len(X_real_sc), len(X_syn_sc)
    if n_syn == 0:
        return ei, ew
    sim = cosine_similarity(X_syn_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_syn):
        s = n_real + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [s, j];  dst += [j, s];  wts += [w, w]
    new_ei = torch.tensor([src, dst], dtype=torch.long)
    new_ew = torch.tensor(wts,        dtype=torch.float)
    aug_ei = torch.cat([ei, new_ei], dim=1)
    aug_ew = torch.cat([ew, new_ew])
    aug_ei, aug_ew = coalesce(aug_ei, aug_ew,
                               num_nodes=n_real + n_syn, reduce='max')
    return aug_ei, aug_ew


def add_val_nodes(ei_aug, ew_aug, X_real_sc, X_val_sc, k=10, n_train_aug=None):
    n_real = len(X_real_sc)
    n_val  = len(X_val_sc)
    if n_train_aug is None:
        n_train_aug = n_real
    sim = cosine_similarity(X_val_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_val):
        v = n_train_aug + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [v, j];  dst += [j, v];  wts += [w, w]
    new_ei  = torch.tensor([src, dst], dtype=torch.long)
    new_ew  = torch.tensor(wts,        dtype=torch.float)
    comb_ei = torch.cat([ei_aug, new_ei],  dim=1)
    comb_ew = torch.cat([ew_aug, new_ew])
    return comb_ei, comb_ew


def _random_walks(edge_index, num_nodes, walk_length, walks_per_node, seed=42):
    import random as _r
    _r.seed(seed)
    adj = [[] for _ in range(num_nodes)]
    ei  = edge_index.cpu().numpy()
    for s, d in zip(ei[0], ei[1]):
        adj[int(s)].append(int(d))
    walks = []
    nodes = list(range(num_nodes))
    for _ in range(walks_per_node):
        _r.shuffle(nodes)
        for start in nodes:
            walk = [start]
            for _ in range(walk_length - 1):
                curr = walk[-1]
                nbrs = adj[curr]
                if nbrs:
                    walk.append(_r.choice(nbrs))
                else:
                    break
            walks.append([str(n) for n in walk])
    return walks


def train_node2vec(edge_index, num_nodes: int, cfg: dict, device):
    walks = _random_walks(edge_index, num_nodes,
                          walk_length    = cfg['N2V_WALK_LEN'],
                          walks_per_node = cfg['N2V_WALKS'],
                          seed           = cfg['SEED'])
    w2v = Word2Vec(sentences   = walks,
                   vector_size = cfg['N2V_DIM'],
                   window      = cfg['N2V_CONTEXT'],
                   min_count   = 0,
                   sg          = 1,
                   workers     = 1,
                   seed        = cfg['SEED'],
                   epochs      = cfg['N2V_EPOCHS'])
    emb = np.zeros((num_nodes, cfg['N2V_DIM']), dtype=np.float32)
    for idx in range(num_nodes):
        key = str(idx)
        if key in w2v.wv:
            emb[idx] = w2v.wv[key]
    return emb


def inductive_n2v(X_new_sc, X_train_sc, n2v_train, k=10):
    sim = cosine_similarity(X_new_sc, X_train_sc)
    out = np.zeros((len(X_new_sc), n2v_train.shape[1]), dtype=np.float32)
    for i in range(len(X_new_sc)):
        top_k = np.argpartition(sim[i], -k)[-k:]
        w     = np.maximum(sim[i][top_k], 0.0)
        wsum  = w.sum()
        w     = w / wsum if wsum > 1e-9 else np.ones(k) / k
        out[i] = (n2v_train[top_k] * w[:, None]).sum(axis=0)
    return out


def apply_smote(X, y, seed=42):
    smote        = SMOTE(random_state=seed, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X, y)
    return X_res, y_res


print("Graph / Node2Vec / SMOTE utilities loaded.")

Graph / Node2Vec / SMOTE utilities loaded.


In [8]:
# CELL 4 — MODEL DEFINITION 

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1   = SAGEConv(in_ch, hidden_ch)
        self.c2   = SAGEConv(hidden_ch, hidden_ch)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)

print("GraphSAGE definition loaded.")

GraphSAGE definition loaded.


In [9]:
# CELL 5 — DATA LOADING
df = pd.read_csv(BASE_CONFIG['DATA_PATH'])
print(f"Loaded  : {df.shape[0]} rows x {df.shape[1]} columns")

y_full        = df[BASE_CONFIG['TARGET_COL']].values.astype(np.int64)
X_full        = df.drop(columns=[BASE_CONFIG['TARGET_COL']]).values.astype(np.float32)
feature_names = df.drop(columns=[BASE_CONFIG['TARGET_COL']]).columns.tolist()
name_to_idx   = {name: i for i, name in enumerate(feature_names)}
print(f"Features: {len(feature_names)}")
print(f"PCOS=0  : {(y_full==0).sum()}   PCOS=1  : {(y_full==1).sum()}")

X_train_pool, _X_test, y_train_pool, _y_test = train_test_split(
    X_full, y_full,
    test_size    = BASE_CONFIG['TEST_SIZE'],
    stratify     = y_full,
    random_state = BASE_CONFIG['SEED'],
)
del _X_test, _y_test
print(f"\nOuter training pool : {len(X_train_pool)} patients  "
      f"(PCOS=0: {(y_train_pool==0).sum()}  PCOS=1: {(y_train_pool==1).sum()})")
print("Test set SEALED — not used in this notebook.")
print(f"Inner CV            : {BASE_CONFIG['INNER_FOLDS']}-fold StratifiedKFold on the training pool")

with open(BASE_CONFIG['SELECTED_FEATURES_PATH']) as f:
    SELECTED = json.load(f)

SUBSET_COLS = {}
print("\nFeature subsets:")
for name in ['PSO', 'HHO', 'MI', 'ALL']:
    feats = SELECTED[name]['features']
    missing = [ft for ft in feats if ft not in name_to_idx]
    if missing:
        raise ValueError(f"{name}: features not found in dataframe columns: {missing}")
    cols = [name_to_idx[ft] for ft in feats]
    SUBSET_COLS[name] = cols
    print(f"  {name:<4}: {len(cols):>2} features")

print("\nData loading and subset resolution complete.")


Loaded  : 541 rows x 49 columns
Features: 48
PCOS=0  : 364   PCOS=1  : 177

Outer training pool : 432 patients  (PCOS=0: 291  PCOS=1: 141)
Test set SEALED — not used in this notebook.
Inner CV            : 5-fold StratifiedKFold on the training pool

Feature subsets:
  PSO : 22 features
  HHO : 17 features
  MI  : 20 features
  ALL : 48 features

Data loading and subset resolution complete.


In [10]:
# CELL 6 — SINGLE-TRIAL OBJECTIVE  

def _run_one_fold(X_ftr, y_ftr, X_fva, y_fva, hidden_dim, dropout, lr,
                  weight_decay, optimiser_name, scheduler_name, cfg, device, fold_seed):
    n_clinical = X_ftr.shape[1]

    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_ftr)
    X_va_sc = scaler.transform(X_fva)

    ei_tr, ew_tr = build_knn_graph(X_tr_sc, k=cfg['K_NEIGHBOURS'])
    n_real = len(X_tr_sc)

    n2v_tr    = train_node2vec(ei_tr, n_real, cfg, device)
    X_tr_full = np.concatenate([X_tr_sc, n2v_tr], axis=1)

    X_tr_sm, y_tr_sm = apply_smote(X_tr_full, y_ftr, cfg['SEED'])
    n_syn       = len(X_tr_sm) - n_real
    n_train_aug = len(X_tr_sm)

    if n_syn > 0:
        X_syn_clin = X_tr_sm[n_real:, :n_clinical]
        ei_aug, ew_aug = add_synthetic_nodes(
            ei_tr, ew_tr, X_tr_sc, X_syn_clin, k=cfg['K_NEIGHBOURS'])
    else:
        ei_aug, ew_aug = ei_tr, ew_tr

    n2v_va    = inductive_n2v(X_va_sc, X_tr_sc, n2v_tr, k=cfg['K_NEIGHBOURS'])
    X_va_full = np.concatenate([X_va_sc, n2v_va], axis=1)

    ei_full, ew_full = add_val_nodes(
        ei_aug, ew_aug, X_tr_sc, X_va_sc,
        k=cfg['K_NEIGHBOURS'], n_train_aug=n_train_aug)

    X_all      = np.vstack([X_tr_sm, X_va_full])
    y_full_arr = np.concatenate([y_tr_sm, y_fva])

    n_total    = len(X_all)
    train_mask = torch.zeros(n_total, dtype=torch.bool)
    val_mask   = torch.zeros(n_total, dtype=torch.bool)
    train_mask[:n_train_aug]                          = True
    val_mask[n_train_aug:n_train_aug + len(X_fva)]    = True

    data = Data(
        x           = torch.tensor(X_all,      dtype=torch.float),
        edge_index  = ei_full,
        edge_weight = ew_full,
        y           = torch.tensor(y_full_arr, dtype=torch.long),
        train_mask  = train_mask,
        val_mask    = val_mask,
    ).to(device)

    n_neg = (y_tr_sm == 0).sum()
    n_pos = (y_tr_sm == 1).sum()
    w_neg = len(y_tr_sm) / (2.0 * n_neg)
    w_pos = len(y_tr_sm) / (2.0 * n_pos)
    cw    = torch.tensor([w_neg, w_pos], dtype=torch.float, device=device)

    in_channels = n_clinical + cfg['N2V_DIM']
    set_seed(fold_seed)                                    # per-fold weight-init determinism
    model = GraphSAGE(in_channels, hidden_dim, 2, dropout).to(device)

    if optimiser_name == 'adamw':
        optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimiser = torch.optim.Adam(model.parameters(),  lr=lr, weight_decay=weight_decay)

    if scheduler_name == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                        optimiser, T_max=cfg['TRIAL_EPOCHS'], eta_min=1e-6)
        plateau = False
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        optimiser, mode='min', factor=0.5, patience=20, min_lr=1e-6)
        plateau = True

    criterion = torch.nn.CrossEntropyLoss(weight=cw)

    for epoch in range(1, cfg['TRIAL_EPOCHS'] + 1):
        model.train()
        optimiser.zero_grad()
        out  = model(data.x, data.edge_index, data.edge_weight)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        if torch.isnan(loss):
            del model, data, optimiser, scheduler, criterion
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            return -1.0                                    # NaN guard
        loss.backward()
        optimiser.step()
        if plateau:
            scheduler.step(loss)
        else:
            scheduler.step()

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_weight)[data.val_mask]
        preds  = logits.argmax(dim=1).cpu().numpy()
        true   = data.y[data.val_mask].cpu().numpy()

    del model, data, optimiser, scheduler, criterion
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    if len(np.unique(preds)) == 1:                         # collapse guard
        return -1.0
    return float(matthews_corrcoef(true, preds))


def run_trial(col_idx, hidden_dim, dropout, lr, weight_decay,
              k_neighbours, n2v_dim, optimiser_name, scheduler_name,
              base_cfg, device):
    set_seed(base_cfg['SEED'])

    cfg = dict(base_cfg)
    cfg['K_NEIGHBOURS'] = int(k_neighbours)
    cfg['N2V_DIM']      = int(n2v_dim)

    Xp = X_train_pool[:, col_idx]
    yp = y_train_pool

    skf = StratifiedKFold(n_splits=base_cfg['INNER_FOLDS'],
                          shuffle=True, random_state=base_cfg['SEED'])
    fold_mccs = []
    for fold_idx, (tr, va) in enumerate(skf.split(Xp, yp)):
        m = _run_one_fold(
            Xp[tr], yp[tr], Xp[va], yp[va],
            hidden_dim, dropout, lr, weight_decay,
            optimiser_name, scheduler_name,
            cfg, device, fold_seed=base_cfg['SEED'] + fold_idx)
        fold_mccs.append(m)

    return float(np.mean(fold_mccs))


print("Trial function defined (regime-aligned).")
print(f"Objective: mean inner k-fold terminal-epoch validation MCC.")
print("NaN / single-class-collapse folds score -1.0.")


Trial function defined (regime-aligned).
Objective: mean inner k-fold terminal-epoch validation MCC.
NaN / single-class-collapse folds score -1.0.


In [11]:
# CELL 7 — OPTUNA OBJECTIVE WRAPPER AND EXPANDED SEARCH SPACE

def make_objective(col_idx, base_cfg, device):
    """
    Search space:
      hidden_dim   : categorical {32, 64, 128}
      dropout      : uniform [0.10, 0.50]
      lr           : log-uniform [1e-4, 1e-2]
      weight_decay : log-uniform [1e-5, 1e-3]
      k_neighbours : integer [5, 20]                 
      n2v_dim      : categorical {16, 32, 64}        
      optimiser    : categorical {adam, adamw}       
      scheduler    : categorical {plateau, cosine}   
    """
    def objective(trial: optuna.Trial) -> float:
        hidden_dim     = trial.suggest_categorical('hidden_dim',   [32, 64, 128])
        dropout        = trial.suggest_float(      'dropout',       0.10, 0.50)
        lr             = trial.suggest_float(      'lr',            1e-4, 1e-2, log=True)
        weight_decay   = trial.suggest_float(      'weight_decay',  1e-5, 1e-3, log=True)
        k_neighbours   = trial.suggest_int(        'k_neighbours',  5,    20)
        n2v_dim        = trial.suggest_categorical('n2v_dim',       [16, 32, 64])
        optimiser_name = trial.suggest_categorical('optimiser',     ['adam', 'adamw'])
        scheduler_name = trial.suggest_categorical('scheduler',     ['plateau', 'cosine'])

        return run_trial(
            col_idx        = col_idx,
            hidden_dim     = hidden_dim,
            dropout        = dropout,
            lr             = lr,
            weight_decay   = weight_decay,
            k_neighbours   = k_neighbours,
            n2v_dim        = n2v_dim,
            optimiser_name = optimiser_name,
            scheduler_name = scheduler_name,
            base_cfg       = base_cfg,
            device         = device,
        )
    return objective


print("Optuna objective wrapper defined.")
print("Search space:")
print("  hidden_dim   : {32, 64, 128}      dropout    : [0.10, 0.50]")
print("  lr           : [1e-4, 1e-2] log   weight_decay: [1e-5, 1e-3] log")
print("  k_neighbours : [5, 20] int        n2v_dim    : {16, 32, 64}")
print("  optimiser    : {adam, adamw}      scheduler  : {plateau, cosine}")

Optuna objective wrapper defined.
Search space:
  hidden_dim   : {32, 64, 128}      dropout    : [0.10, 0.50]
  lr           : [1e-4, 1e-2] log   weight_decay: [1e-5, 1e-3] log
  k_neighbours : [5, 20] int        n2v_dim    : {16, 32, 64}
  optimiser    : {adam, adamw}      scheduler  : {plateau, cosine}


In [12]:
# CELL 8 — RUN ENHANCED OPTUNA SEARCH FOR EACH FEATURE SUBSET

SUBSET_ORDER = ['PSO', 'HHO', 'MI', 'ALL']
enhanced_results = {}
OUT = BASE_CONFIG['OUTPUT_DIR']

for subset in SUBSET_ORDER:
    col_idx  = SUBSET_COLS[subset]
    n_trials = TRIALS_PER_SUBSET[subset]

    print(f"\n{'='*64}")
    print(f"  Enhanced Optuna: GraphSAGE + {subset}  "
          f"({len(col_idx)} features, {n_trials} trials)")
    print(f"{'='*64}")

    study = optuna.create_study(
        direction  = 'maximize',
        study_name = f'pcos_graphsage_{subset.lower()}_enhanced',
        sampler    = optuna.samplers.TPESampler(seed=GLOBAL_SEED),
        pruner     = optuna.pruners.MedianPruner(n_warmup_steps=10),
    )

    study.enqueue_trial({
        'hidden_dim'   : 64,
        'dropout'      : 0.3,
        'lr'           : 1e-3,
        'weight_decay' : 1e-4,
        'k_neighbours' : 10,
        'n2v_dim'      : 64,
        'optimiser'    : 'adam',
        'scheduler'    : 'plateau',
    })

    completed = [0]
    def progress_callback(study, trial, _n=n_trials):
        completed[0] += 1
        status = f"MCC={trial.value:.4f}" if trial.value is not None else "pruned"
        best   = study.best_value if len(study.trials) > 0 else float('nan')
        print(f"  Trial {completed[0]:>3}/{_n}  {status}   best={best:.4f}   "
              f"params={trial.params}")

    study.optimize(
        make_objective(col_idx, BASE_CONFIG, DEVICE),
        n_trials          = n_trials,
        callbacks         = [progress_callback],
        gc_after_trial    = True,
        show_progress_bar = False,
    )

    best = study.best_trial
    print(f"\n  * {subset} best trial #{best.number}   MCC {best.value:.4f}")
    for kk, vv in best.params.items():
        print(f"    {kk:<14}: {vv}")

    enhanced_results[subset] = {
        'n_features'  : len(col_idx),
        'best_mcc'    : best.value,
        'best_trial'  : best.number,
        'params'      : best.params,
        'all_trials'  : [
            {'number': t.number, 'value': t.value, 'params': t.params}
            for t in study.trials if t.value is not None
        ],
    }

    out_path = os.path.join(OUT, 'best_hyperparams_graphsage_enhanced.json')
    with open(out_path, 'w') as f:
        json.dump(enhanced_results, f, indent=2)
    print(f"  Saved -> {out_path}")

print(f"\n{'='*64}")
print("  Enhanced Optuna search complete for all four subsets.")
print(f"{'='*64}")


  Enhanced Optuna: GraphSAGE + PSO  (22 features, 50 trials)
  Trial   1/50  MCC=0.7205   best=0.7205   params={'hidden_dim': 64, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'k_neighbours': 10, 'n2v_dim': 64, 'optimiser': 'adam', 'scheduler': 'plateau'}
  Trial   2/50  MCC=0.7113   best=0.7205   params={'hidden_dim': 64, 'dropout': 0.3394633936788146, 'lr': 0.0002051338263087451, 'weight_decay': 2.0511104188433963e-05, 'k_neighbours': 5, 'n2v_dim': 16, 'optimiser': 'adamw', 'scheduler': 'plateau'}
  Trial   3/50  MCC=0.7696   best=0.7696   params={'hidden_dim': 128, 'dropout': 0.3099025726528951, 'lr': 0.0007309539835912913, 'weight_decay': 3.8234752246751835e-05, 'k_neighbours': 14, 'n2v_dim': 64, 'optimiser': 'adamw', 'scheduler': 'cosine'}
  Trial   4/50  MCC=0.7226   best=0.7696   params={'hidden_dim': 128, 'dropout': 0.16820964947491662, 'lr': 0.00013492834268013249, 'weight_decay': 0.000790261954970823, 'k_neighbours': 20, 'n2v_dim': 16, 'optimiser': 'adam', 'scheduler'

In [13]:
# CELL 9 — RESULTS SUMMARY TABLE

print("\n" + "="*94)
print(f"  {'Subset':<7} {'#Feat':>5}  {'CV-MCC':>9}  {'hidden':>7}  {'drop':>6}  "
      f"{'lr':>9}  {'wd':>9}  {'k':>3}  {'n2v':>4}  {'optim':>6}  {'sched':>8}")
print("="*94)
for subset in SUBSET_ORDER:
    r = enhanced_results[subset]; p = r['params']
    print(f"  {subset:<7} {r['n_features']:>5}  {r['best_mcc']:>9.4f}  "
          f"{p['hidden_dim']:>7}  {p['dropout']:>6.3f}  {p['lr']:>9.6f}  "
          f"{p['weight_decay']:>9.6f}  {p['k_neighbours']:>3}  {p['n2v_dim']:>4}  "
          f"{p['optimiser']:>6}  {p['scheduler']:>8}")
print("="*94)
print()
print("NOTE: CV-MCC is the mean inner k-fold terminal-epoch validation MCC used as the")
print("      Optuna fitness surface. It is NOT the sealed-test result and does not")
print("      decide the final Tier-A ranking. The combined evaluation notebook")
print("      evaluates each subset once on the sealed test set for the comparison.")
print()
print(f"Saved -> {os.path.join(BASE_CONFIG['OUTPUT_DIR'], 'best_hyperparams_graphsage_enhanced.json')}")


  Subset  #Feat     CV-MCC   hidden    drop         lr         wd    k   n2v   optim     sched
  PSO        22     0.7829       32   0.404   0.001484   0.000455   12    64   adamw    cosine
  HHO        17     0.7738       64   0.351   0.000245   0.000076   10    64    adam   plateau
  MI         20     0.7712       32   0.498   0.002473   0.000265   20    64    adam    cosine
  ALL        48     0.7296      128   0.374   0.000781   0.000014    8    64   adamw    cosine

NOTE: CV-MCC is the mean inner k-fold terminal-epoch validation MCC used as the
      Optuna fitness surface. It is NOT the sealed-test result and does not
      decide the final Tier-A ranking. The combined evaluation notebook
      evaluates each subset once on the sealed test set for the comparison.

Saved -> /kaggle/working/best_hyperparams_graphsage_enhanced.json


In [14]:
# CELL 10 — VERIFY SAVED JSON
out_path = os.path.join(BASE_CONFIG['OUTPUT_DIR'], 'best_hyperparams_graphsage_enhanced.json')
with open(out_path) as f:
    loaded = json.load(f)

print("Verification — best_hyperparams_graphsage_enhanced.json:")
for subset, result in loaded.items():
    print(f"\n  {subset}  ({result['n_features']} features)")
    print(f"    best_mcc (inner) : {result['best_mcc']:.4f}")
    for kk, vv in result['params'].items():
        print(f"    {kk:<14}: {vv}")

Verification — best_hyperparams_graphsage_enhanced.json:

  PSO  (22 features)
    best_mcc (inner) : 0.7829
    hidden_dim    : 32
    dropout       : 0.4041521218793736
    lr            : 0.0014835151621091199
    weight_decay  : 0.00045455550610998925
    k_neighbours  : 12
    n2v_dim       : 64
    optimiser     : adamw
    scheduler     : cosine

  HHO  (17 features)
    best_mcc (inner) : 0.7738
    hidden_dim    : 64
    dropout       : 0.35130945117725687
    lr            : 0.0002446001779697765
    weight_decay  : 7.635295685715325e-05
    k_neighbours  : 10
    n2v_dim       : 64
    optimiser     : adam
    scheduler     : plateau

  MI  (20 features)
    best_mcc (inner) : 0.7712
    hidden_dim    : 32
    dropout       : 0.4979696114920618
    lr            : 0.002473267015708714
    weight_decay  : 0.0002654334209893315
    k_neighbours  : 20
    n2v_dim       : 64
    optimiser     : adam
    scheduler     : cosine

  ALL  (48 features)
    best_mcc (inner) : 0.7296
 